# KYC — Identity Document Extraction Pipeline
### Local Qwen-VL on Domino Data Lab · forensic (anti-hallucination) OCR

**Model (local, offline):** `/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B/main`

---

## Pipeline

| # | Stage | Output |
|---|-------|--------|
| 1 | Unzip `kyc_documents.zip`, keep only the 5 target PDFs | `extracted/<customer>/…` |
| 2 | Presence / absence check of the 5 documents per customer | `reports/document_presence.{csv,xlsx,json}` |
| 3 | Target-customer list (those having `JUSTIFICATIF IDENTITE.PDF`) | `reports/target_customers.{csv,json}` |
| 4 | Rasterise → orientation fix → deskew → enhance → quality metrics | `pages/<customer>/p###.jpg` |
| 5 | Page triage (skip blank / non-ID pages) | ranked page list |
| 6 | Batched VLM extraction, JSON-schema-constrained, greedy | `extractions/<customer>/identity_extraction.json` |
| 7 | Multi-page merge by **visual confidence**, conflict → `null` | consolidated JSONL + CSV |

## Non-negotiable rules encoded in this pipeline
* Greedy decoding (`temperature=0`) — no sampling, no creativity.
* Grammar-constrained JSON — the model **cannot** emit free text.
* No post-hoc "repair": MRZ checksums, date formats and document-number patterns are
  **validated and flagged only**, never corrected.
* Conflicting values across pages at equal confidence ⇒ `null` + flagged for human review.
* Any parsing/inference failure ⇒ all-`null` skeleton, never a partial guess.

## Latency strategy (lowest possible inference time)
1. **One engine load** for the whole run; **prefix caching on** — the long system prompt is
   encoded once and reused for every page.
2. **Full-corpus batching** — all candidate pages of all customers go into a single
   `generate()` call, so vLLM's continuous batching keeps the GPU saturated.
3. **Page triage** — blank / non-identity pages never reach the GPU.
4. **Two waves** — wave 1 processes the top-`K` ranked pages only; wave 2 retries
   (rotated 180°, higher resolution) *only* the customers whose core fields are still missing.
5. **Pixel budget** — pages are capped to `max_pixels` before tokenisation (vision tokens
   dominate latency), with an automatic high-res retry for low-confidence cases.
6. **CPU/GPU overlap** — rasterisation and pre-processing run in a process pool.

## 0 · Environment

In [ ]:
# ── Python packages ──────────────────────────────────────────────────────────
# Air-gapped Domino: add  -i https://<internal-pypi>/simple  --trusted-host <host>
%pip install -q \
    "vllm>=0.9.0" \
    "transformers>=4.57.0" \
    "accelerate>=1.0.0" \
    "qwen-vl-utils>=0.0.11" \
    "pypdfium2>=4.30.0" \
    "pdf2image>=1.17.0" \
    "opencv-python-headless>=4.10.0" \
    "pillow>=10.4.0" \
    "numpy>=1.26,<2.3" \
    "pandas>=2.2" \
    "openpyxl>=3.1" \
    "pydantic>=2.8" \
    "jsonschema>=4.23" \
    "pytesseract>=0.3.13" \
    "tqdm>=4.66"

In [ ]:
# ── OS packages (OPTIONAL) ───────────────────────────────────────────────────
# Only needed for (a) the Tesseract orientation/triage pass and (b) the pdf2image
# fallback renderer. The pipeline degrades gracefully if they are absent.
# Skip this cell if you have no sudo in the Domino workspace.
#
# !sudo apt-get update -qq && sudo apt-get install -y -qq \
#      tesseract-ocr tesseract-ocr-fra tesseract-ocr-ara tesseract-ocr-eng poppler-utils

In [ ]:
import os

# Force fully-offline operation: nothing must be fetched from the internet.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")
# Uncomment and adapt if the workspace exposes several GPUs and you want a subset:
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
print("offline env set")

## 1 · Configuration

Everything tunable lives here. `max_pixels` and `top_k_pages` are the two knobs that
dominate inference time.

In [ ]:
from dataclasses import dataclass, field
from pathlib import Path

try:
    import torch
    _N_GPU = torch.cuda.device_count()
except Exception:                                   # CPU-only sanity runs
    _N_GPU = 0


@dataclass
class Config:
    # ── I/O ──────────────────────────────────────────────────────────────────
    zip_path: Path = Path("/mnt/data/kyc_documents.zip")
    work_dir: Path = Path("/mnt/data/kyc_run")

    # ── Model / backend ──────────────────────────────────────────────────────
    model_path: str = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B/main"
    backend: str = "vllm"                 # "vllm" (fast) | "transformers" (fallback)
    tensor_parallel_size: int = max(1, _N_GPU)
    gpu_memory_utilization: float = 0.90
    max_model_len: int = 8192
    max_new_tokens: int = 640             # the JSON skeleton needs ~250-400 tokens
    dtype: str = "bfloat16"
    enable_prefix_caching: bool = True    # system prompt encoded once → big win
    guided_json: bool = True              # grammar-constrained decoding

    # ── Vision token budget (latency driver) ─────────────────────────────────
    #   28*28 = one Qwen vision patch. 1280 patches ≈ 1.0 Mpx ≈ ~1300 vision tokens.
    min_pixels: int = 256 * 28 * 28
    max_pixels: int = 1280 * 28 * 28
    max_pixels_retry: int = 2048 * 28 * 28   # wave-2 high-resolution retry

    # ── Rasterisation / pre-processing ───────────────────────────────────────
    render_dpi: int = 250
    render_dpi_retry: int = 350
    max_pages_per_pdf: int = 12
    jpeg_quality: int = 92
    n_cpu_workers: int = min(16, (os.cpu_count() or 4))
    # Threads by default: functions defined in a notebook cell are not picklable, so a
    # process pool silently falls back to serial. pdfium/OpenCV release the GIL, so
    # threads give the real speed-up here. Set True only if you move this to a .py module.
    use_multiprocessing: bool = False
    do_deskew: bool = True
    do_enhance: bool = True
    use_tesseract_osd: bool = True        # 0/90/180/270 detection
    use_tesseract_triage: bool = True     # cheap keyword ranking of pages

    # ── Triage / waves ───────────────────────────────────────────────────────
    top_k_pages: int = 2                  # wave-1 pages sent to the GPU per customer
    blank_ink_threshold: float = 0.004     # < 0.4 % ink ⇒ page considered blank
    enable_wave2_retry: bool = True

    # ── Target documents (normalised aliases → canonical key) ────────────────
    doc_aliases: dict = field(default_factory=lambda: {
        "JUSTIFICATIF_IDENTITE": [
            "JUSTIFICATIF IDENTITE", "JUSTIFICATIF D IDENTITE", "JUSTIFICATIF DIDENTITE",
            "JUSTIFICATIF DE IDENTITE", "PIECE IDENTITE", "PIECE D IDENTITE",
        ],
        "JUSTIFICATIF_DOMICILE": [
            "JUSTIFICATIF DOMICILE", "JUSTIFICATIF DE DOMICILE", "JUSTIF DOMICILE",
        ],
        "CONVENTION_COMPTE": [
            "CONVENTION COMPTE", "CONVENTION DE COMPTE", "CONVENTION DU COMPTE",
        ],
        "FATCA": ["FATCA", "FORMULAIRE FATCA", "FATCA CRS"],
        "CARTON_SIGNATURE": [
            "CARTON SIGNATUTE", "CARTON SIGNATURE", "CARTON DE SIGNATURE",
            "CARTON DE SIGNATUTE", "SPECIMEN SIGNATURE",
        ],
    })

    def __post_init__(self):
        self.zip_path = Path(self.zip_path)
        self.work_dir = Path(self.work_dir)
        self.extracted_dir = self.work_dir / "extracted"
        self.pages_dir     = self.work_dir / "pages"
        self.reports_dir   = self.work_dir / "reports"
        self.out_dir       = self.work_dir / "extractions"
        self.debug_dir     = self.work_dir / "_debug"
        for d in (self.extracted_dir, self.pages_dir, self.reports_dir,
                  self.out_dir, self.debug_dir):
            d.mkdir(parents=True, exist_ok=True)


CFG = Config()
DOC_KEYS = list(CFG.doc_aliases.keys())
IDENTITY_KEY = "JUSTIFICATIF_IDENTITE"

print(f"work_dir : {CFG.work_dir}")
print(f"model    : {CFG.model_path}")
print(f"GPUs     : {_N_GPU}  |  CPU workers: {CFG.n_cpu_workers}")

In [ ]:
import json, logging, re, shutil, time, unicodedata, zipfile
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from typing import Any, Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger("kyc")
logging.getLogger("vllm").setLevel(logging.WARNING)


def normalize_name(s: str) -> str:
    """Upper-case, accent-free, punctuation-free form used for filename matching."""
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.upper()
    s = re.sub(r"\.PDF$", "", s)
    s = re.sub(r"[^A-Z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


_ALIAS_LOOKUP = {normalize_name(a): k for k, al in CFG.doc_aliases.items() for a in al}


def match_doc_type(filename: str) -> Optional[str]:
    """Map a filename to one of the 5 canonical document keys (None = not a target)."""
    n = normalize_name(filename)
    if n in _ALIAS_LOOKUP:
        return _ALIAS_LOOKUP[n]
    # tolerant containment match for names with prefixes/suffixes (scan ids, dates…)
    for alias, key in _ALIAS_LOOKUP.items():
        if alias in n:
            return key
    return None


assert match_doc_type("JUSTIFICATIF IDENTITE.PDF") == "JUSTIFICATIF_IDENTITE"
assert match_doc_type("Carton Signatute.pdf")      == "CARTON_SIGNATURE"
assert match_doc_type("RIB.pdf") is None
print("helpers ready")

## 2 · Unzip — keep only the 5 target documents

Path traversal (`../`) and absolute members are rejected. The customer identifier is the
first path component below the archive's common root, so nested scan folders are handled.

In [ ]:
def _common_root(paths: List[str]) -> int:
    """Number of leading path components shared by every member (archive wrapper dir)."""
    if not paths:
        return 0
    split = [p.split("/") for p in paths]
    n = 0
    while True:
        if any(len(s) <= n + 1 for s in split):
            return n
        first = split[0][n]
        if any(s[n] != first for s in split):
            return n
        n += 1


def _is_safe(member: str) -> bool:
    p = Path(member)
    return not p.is_absolute() and ".." not in p.parts


def unzip_kyc(cfg: Config) -> Tuple[Dict[str, Dict[str, str]], Dict[str, List[str]]]:
    """Extract only the 5 target PDFs.

    Returns
    -------
    inventory : {customer_id: {doc_key: extracted_path}}
    ignored   : {customer_id: [filenames not in the target list]}
    """
    inventory: Dict[str, Dict[str, str]] = defaultdict(dict)
    ignored: Dict[str, List[str]] = defaultdict(list)

    with zipfile.ZipFile(cfg.zip_path) as zf:
        members = [m for m in zf.namelist() if not m.endswith("/")]
        members = [m for m in members if _is_safe(m)]
        root = _common_root(members)

        for m in tqdm(members, desc="unzip", unit="file"):
            parts = m.split("/")[root:]
            if len(parts) < 2:
                continue                                  # loose file, no customer folder
            customer, filename = parts[0], parts[-1]
            inventory.setdefault(customer, {})

            key = match_doc_type(filename) if filename.lower().endswith(".pdf") else None
            if key is None:
                ignored[customer].append(filename)
                continue
            if key in inventory[customer]:                # duplicate → keep first, log
                ignored[customer].append(f"{filename} (duplicate of {key})")
                continue

            dest = cfg.extracted_dir / customer / f"{key}.pdf"
            dest.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(m) as src, open(dest, "wb") as out:
                shutil.copyfileobj(src, out)
            inventory[customer][key] = str(dest)

    return dict(inventory), dict(ignored)


t0 = time.time()
INVENTORY, IGNORED = unzip_kyc(CFG)
log.info("%d customer folders — extraction done in %.1fs", len(INVENTORY), time.time() - t0)

## 3 · Presence / absence report

In [ ]:
def build_presence_report(cfg: Config, inventory, ignored) -> pd.DataFrame:
    rows = []
    for cust in sorted(inventory):
        found = inventory[cust]
        row = {"customer_id": cust}
        for k in DOC_KEYS:
            row[k] = k in found
        row["n_present"] = sum(bool(row[k]) for k in DOC_KEYS)
        row["n_missing"] = len(DOC_KEYS) - row["n_present"]
        row["missing_documents"] = ";".join(k for k in DOC_KEYS if not row[k])
        row["folder_complete"] = row["n_missing"] == 0
        row["is_target_customer"] = row[IDENTITY_KEY]
        row["other_files_ignored"] = len(ignored.get(cust, []))
        rows.append(row)

    df = pd.DataFrame(rows).sort_values("customer_id").reset_index(drop=True)
    df.to_csv(cfg.reports_dir / "document_presence.csv", index=False)
    try:
        df.to_excel(cfg.reports_dir / "document_presence.xlsx", index=False)
    except Exception as e:                                # openpyxl missing
        log.warning("xlsx skipped: %s", e)
    (cfg.reports_dir / "document_presence.json").write_text(
        json.dumps(df.to_dict(orient="records"), indent=2, ensure_ascii=False), encoding="utf-8")
    (cfg.reports_dir / "ignored_files.json").write_text(
        json.dumps(ignored, indent=2, ensure_ascii=False), encoding="utf-8")
    return df


PRESENCE = build_presence_report(CFG, INVENTORY, IGNORED)

print(f"customers              : {len(PRESENCE)}")
print(f"complete folders (5/5) : {int(PRESENCE.folder_complete.sum())}")
print("\nper-document availability")
print(PRESENCE[DOC_KEYS].sum().to_frame("present").assign(
    missing=lambda d: len(PRESENCE) - d["present"]))
PRESENCE.head(10)

## 4 · Target customers = folders containing `JUSTIFICATIF IDENTITE.PDF`

In [ ]:
TARGETS = {c: INVENTORY[c][IDENTITY_KEY] for c in sorted(INVENTORY)
           if IDENTITY_KEY in INVENTORY[c]}

pd.DataFrame({"customer_id": list(TARGETS), "identity_pdf": list(TARGETS.values())}) \
  .to_csv(CFG.reports_dir / "target_customers.csv", index=False)
(CFG.reports_dir / "target_customers.json").write_text(
    json.dumps(sorted(TARGETS), indent=2), encoding="utf-8")

print(f"target customers : {len(TARGETS)} / {len(INVENTORY)}")
print(f"excluded (no identity document) : {len(INVENTORY) - len(TARGETS)}")
list(TARGETS)[:10]

## 5 · Rasterisation + image restoration

`pypdfium2` is used as the renderer (3–5× faster than poppler and no external binary),
with `pdf2image` as a fallback. Restoration order matters:

`render → coarse rotation (0/90/180/270) → deskew (±8°) → CLAHE + unsharp → pixel budget`

> Enhancement is applied to improve *legibility only*. It never licenses a higher
> confidence: the model still reports confidence from what it can actually see, and a
> character that is destroyed by the scan stays unreadable.

In [ ]:
import pypdfium2 as pdfium


def render_pdf_to_disk(pdf_path: str, out_dir: str, dpi: int, max_pages: int,
                       jpeg_quality: int = 92) -> List[str]:
    """Rasterise a PDF to JPEG pages. Returns the list of written page paths."""
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    paths = []
    try:
        pdf = pdfium.PdfDocument(pdf_path)
        n = min(len(pdf), max_pages)
        scale = dpi / 72.0
        for i in range(n):
            img = pdf[i].render(scale=scale).to_pil().convert("RGB")
            p = out / f"p{i:03d}.jpg"
            img.save(p, "JPEG", quality=jpeg_quality, optimize=True)
            paths.append(str(p))
        pdf.close()
    except Exception as e:                                  # corrupted / exotic PDF
        try:
            from pdf2image import convert_from_path
            for i, img in enumerate(convert_from_path(pdf_path, dpi=dpi)[:max_pages]):
                p = out / f"p{i:03d}.jpg"
                img.convert("RGB").save(p, "JPEG", quality=jpeg_quality)
                paths.append(str(p))
        except Exception as e2:
            return [f"__ERROR__:{e} | fallback:{e2}"]
    return paths

In [ ]:
# ── Quality metrics ──────────────────────────────────────────────────────────
def quality_metrics(bgr: np.ndarray) -> Dict[str, Any]:
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    blur = float(cv2.Laplacian(gray, cv2.CV_64F).var())      # low  ⇒ blurred
    contrast = float(gray.std())                             # low  ⇒ washed out
    ink = float((cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY_INV, 35, 15) > 0).mean())
    reasons = []
    if min(h, w) < 700:    reasons.append("low_resolution")
    if blur < 60:          reasons.append("blur")
    if contrast < 28:      reasons.append("low_contrast")
    if ink < 0.004:        reasons.append("near_blank")
    return {"width": w, "height": h, "blur_var": round(blur, 1),
            "contrast_std": round(contrast, 1), "ink_ratio": round(ink, 4),
            "flags": reasons}


# ── Coarse orientation: 0 / 90 / 180 / 270 ───────────────────────────────────
_TESS_OK = None


def _tesseract_available() -> bool:
    global _TESS_OK
    if _TESS_OK is None:
        try:
            import pytesseract
            pytesseract.get_tesseract_version()
            _TESS_OK = True
        except Exception:
            _TESS_OK = False
            log.warning("Tesseract unavailable → OSD/triage disabled "
                        "(wave-2 180° retry compensates)")
    return _TESS_OK


def detect_rotation(bgr: np.ndarray, cfg: Config) -> int:
    """Return the clockwise rotation (deg) that must be undone. 0 if unknown."""
    if not (cfg.use_tesseract_osd and _tesseract_available()):
        return 0
    try:
        import pytesseract
        small = bgr
        if max(small.shape[:2]) > 1600:
            s = 1600 / max(small.shape[:2])
            small = cv2.resize(small, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        osd = pytesseract.image_to_osd(cv2.cvtColor(small, cv2.COLOR_BGR2RGB),
                                       config="--psm 0 --oem 1",
                                       output_type=pytesseract.Output.DICT)
        conf = float(osd.get("orientation_conf", 0) or 0)
        rot = int(osd.get("rotate", 0) or 0)
        return rot if conf >= 1.0 else 0
    except Exception:
        return 0


def apply_rotation(bgr: np.ndarray, deg: int) -> np.ndarray:
    return {0: bgr,
            90: cv2.rotate(bgr, cv2.ROTATE_90_COUNTERCLOCKWISE),
            180: cv2.rotate(bgr, cv2.ROTATE_180),
            270: cv2.rotate(bgr, cv2.ROTATE_90_CLOCKWISE)}.get(deg % 360, bgr)


# ── Fine deskew: maximise horizontal projection sharpness ────────────────────
def _proj_score(bin_small: np.ndarray) -> float:
    proj = bin_small.sum(axis=1, dtype=np.float64)
    return float(((proj[1:] - proj[:-1]) ** 2).mean())


def _rotate_keep(img: np.ndarray, angle: float) -> np.ndarray:
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_REPLICATE)


def deskew(bgr: np.ndarray, max_angle: float = 8.0) -> Tuple[np.ndarray, float]:
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    s = 800 / max(gray.shape)
    small = cv2.resize(gray, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else gray
    binr = (cv2.adaptiveThreshold(small, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY_INV, 25, 12) > 0).astype(np.float32)
    best, best_a = _proj_score(binr), 0.0
    for a in np.arange(-max_angle, max_angle + 0.01, 1.0):    # coarse
        sc = _proj_score(_rotate_keep(binr, a))
        if sc > best:
            best, best_a = sc, float(a)
    for a in np.arange(best_a - 1.0, best_a + 1.01, 0.25):    # fine
        sc = _proj_score(_rotate_keep(binr, a))
        if sc > best:
            best, best_a = sc, float(a)
    if abs(best_a) < 0.2:
        return bgr, 0.0
    return _rotate_keep(bgr, best_a), round(best_a, 2)


# ── Legibility enhancement (never a confidence booster) ──────────────────────
def enhance(bgr: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    out = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)
    blur = cv2.GaussianBlur(out, (0, 0), 1.2)
    return cv2.addWeighted(out, 1.5, blur, -0.5, 0)


def fit_pixel_budget(pil: Image.Image, max_pixels: int) -> Image.Image:
    w, h = pil.size
    if w * h <= max_pixels:
        return pil
    s = (max_pixels / (w * h)) ** 0.5
    return pil.resize((max(28, int(w * s)), max(28, int(h * s))), Image.LANCZOS)


def preprocess_page(path: str, cfg: Config) -> Tuple[Image.Image, Dict[str, Any]]:
    bgr = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"unreadable image: {path}")
    meta = {"page_path": path}
    rot = detect_rotation(bgr, cfg)
    if rot:
        bgr = apply_rotation(bgr, rot)
    meta["rotation_applied"] = rot
    if cfg.do_deskew:
        bgr, ang = deskew(bgr)
        meta["skew_corrected_deg"] = ang
    meta["quality"] = quality_metrics(bgr)
    if cfg.do_enhance:
        bgr = enhance(bgr)
    pil = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    return pil, meta


print("pre-processing ready")

## 6 · Page triage

Identity scans routinely contain blank versos, envelopes and cover sheets. Triage keeps
them off the GPU. Ranking uses a very cheap Tesseract pass (~150 ms/page) when available,
otherwise edge/ink density. **Triage only decides *which pixels* are shown to the model —
it never contributes a character to the output.**

In [ ]:
ID_KEYWORDS = [
    "PASSPORT", "PASSEPORT", "CARTE NATIONALE", "IDENTITE", "IDENTITY", "CARD",
    "REPUBLIQUE", "REPUBLIC", "ROYAUME", "KINGDOM", "PERMIS", "LICENCE", "LICENSE",
    "SEJOUR", "RESIDENCE", "NATIONALITE", "NATIONALITY", "SURNAME", "NOM", "PRENOM",
    "GIVEN", "BIRTH", "NAISSANCE", "EXPIR", "DELIVR", "ISSUE", "AUTORITE", "AUTHORITY",
    "CIN", "CNI", "NNI", "بطاقة", "هوية", "جواز", "المملكة", "الجمهورية",
]
MRZ_RE = re.compile(r"[A-Z0-9<]{25,}")


def _cheap_text(bgr: np.ndarray) -> str:
    if not _tesseract_available():
        return ""
    try:
        import pytesseract
        s = 1100 / max(bgr.shape[:2])
        small = cv2.resize(bgr, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else bgr
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        return pytesseract.image_to_string(gray, lang="eng+fra", config="--psm 6 --oem 1")
    except Exception:
        return ""


def score_page(path: str, cfg: Config) -> Dict[str, Any]:
    bgr = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        return {"page_path": path, "score": -1.0, "blank": True, "reason": "unreadable"}
    q = quality_metrics(bgr)
    if q["ink_ratio"] < cfg.blank_ink_threshold:
        return {"page_path": path, "score": -1.0, "blank": True, "reason": "blank",
                "quality": q}

    score = 1.0 + min(q["ink_ratio"] * 10, 2.0)
    hits, has_mrz = 0, False
    if cfg.use_tesseract_triage:
        txt = normalize_name(_cheap_text(bgr))
        hits = sum(1 for k in ID_KEYWORDS if normalize_name(k) and normalize_name(k) in txt)
        has_mrz = bool(MRZ_RE.search(txt.replace(" ", "")))
        score += 2.0 * hits + (6.0 if has_mrz else 0.0)
    return {"page_path": path, "score": round(float(score), 3), "blank": False,
            "keyword_hits": hits, "mrz_like": has_mrz, "quality": q}


def prepare_customer(customer: str, pdf_path: str, cfg: Config,
                     dpi: Optional[int] = None) -> Dict[str, Any]:
    """Rasterise + triage one customer's identity document."""
    out_dir = cfg.pages_dir / customer
    pages = render_pdf_to_disk(pdf_path, str(out_dir), dpi or cfg.render_dpi,
                               cfg.max_pages_per_pdf, cfg.jpeg_quality)
    if pages and isinstance(pages[0], str) and pages[0].startswith("__ERROR__"):
        return {"customer_id": customer, "error": pages[0], "pages": []}
    scored = [score_page(p, cfg) for p in pages]
    ranked = sorted([s for s in scored if not s["blank"]], key=lambda d: -d["score"])

    blank_fallback = False
    if not ranked and scored:
        # SAFETY NET: triage must never silently discard a whole identity document.
        # A small card photographed on a large white background can fall under the ink
        # threshold. When every page looks blank, send the densest ones anyway and let
        # the model be the one to say "unreadable".
        blank_fallback = True
        ranked = sorted(scored,
                        key=lambda d: -(d.get("quality", {}).get("ink_ratio", 0)))[:3]

    return {"customer_id": customer, "n_pages": len(pages),
            "n_blank": sum(s["blank"] for s in scored),
            "blank_fallback": blank_fallback, "pages": ranked}


def prepare_all(cfg: Config, targets: Dict[str, str]) -> Dict[str, Dict[str, Any]]:
    items = list(targets.items())
    results = {}
    Pool = ProcessPoolExecutor if cfg.use_multiprocessing else ThreadPoolExecutor
    try:
        with Pool(max_workers=cfg.n_cpu_workers) as ex:
            futs = {ex.submit(prepare_customer, c, p, cfg): c for c, p in items}
            for f in tqdm(as_completed(futs), total=len(futs), desc="rasterise+triage"):
                results[futs[f]] = f.result()
    except Exception as e:                                     # pool unavailable
        log.warning("parallel prep failed (%s) → serial", e)
        for c, p in tqdm(items, desc="rasterise+triage (serial)"):
            results[c] = prepare_customer(c, p, cfg)
    return results


t0 = time.time()
PREPARED = prepare_all(CFG, TARGETS)
log.info("prepared %d documents (%d pages) in %.1fs",
         len(PREPARED),
         sum(p.get("n_pages", 0) for p in PREPARED.values()),
         time.time() - t0)

## 7 · Prompts and output contract

The system prompt is the forensic-transcription contract, sent verbatim on every request.
With prefix caching enabled it is encoded **once** for the entire run.

In [ ]:
SYSTEM_PROMPT = """You are a high-precision OCR and document extraction engine specialized in identity documents such as passports, national identity cards, residence permits, and driver's licenses.

Your primary objective is to extract ONLY information that is visually supported by the supplied document image.

CRITICAL ANTI-HALLUCINATION RULES

NEVER guess a character, digit, word, name, date, or document number.
NEVER infer missing characters from context.
NEVER complete partially visible text.
NEVER correct spelling, transliteration, formatting, or apparent OCR errors.
NEVER use external knowledge to reconstruct unreadable text.
NEVER assume what a field "should" contain based on the document type.
If a character cannot be distinguished reliably from the image, mark that character as uncertain.
If a complete field cannot be read reliably, return null.
A partially unreadable value is preferable to an invented complete value.
Do NOT manufacture a value merely because the requested JSON field requires a value.
Do NOT answer with explanations, guesses, or conversational text.
The image is the ONLY authoritative source of the extracted value.

VISUAL EVIDENCE RULE

For every extracted value, ask internally:
"Can I directly see every character of this value in the image?"
If the answer is NO:
- Do not guess.
- Do not infer.
- Do not reconstruct.
- Return the value as null or mark the uncertain character(s).

For example, if the image appears to contain A12?45B7 and the fourth character cannot be reliably distinguished, DO NOT output A12345B7. Instead return the value as null with confidence "low" and list the 1-based index of each unreadable character in "uncertain_positions".

CHARACTER-LEVEL TRANSCRIPTION

Preserve exactly what is visually present.
Do not: fix spelling, normalize names, translate names, expand abbreviations, change Arabic names into a preferred Latin spelling, or replace visually similar characters unless the image clearly establishes the character.

Pay particular attention to visually similar characters:
0 / O ; 1 / I / L ; 2 / Z ; 5 / S ; 6 / G ; 8 / B ; C / G ; U / V ; D / O ; M / N
If the image does not allow a reliable distinction, mark the character as uncertain.

DOCUMENT FIELDS

Extract only the fields that are requested. Do not invent fields that are not visible.

MRZ

If the document contains a Machine Readable Zone:
- Transcribe the MRZ exactly as visible.
- Do not reconstruct missing characters.
- Preserve < characters.
- Do not "repair" the MRZ simply because the expected ICAO format suggests another character.
- If a character is unreadable, mark it as uncertain.
- If the MRZ is too degraded to reliably transcribe, return null.
- MRZ validation may identify an inconsistency but must NEVER be used to invent a missing character.

IMAGE QUALITY

Before extraction, evaluate: resolution, blur, compression artifacts, contrast, skew, cropping, shadows, missing portions, character visibility.
If image quality prevents reliable extraction, report that instead of guessing.
Do not assume that image enhancement makes an unreadable character readable.

FIELD-LEVEL CONFIDENCE

For each field assign one of:
- high: every character is clearly visible
- medium: most characters are visible but one or more are somewhat ambiguous
- low: significant ambiguity exists
- unreadable: the value cannot be reliably extracted

Confidence refers to VISUAL EVIDENCE, not how plausible the resulting value appears. A plausible value with weak visual evidence must NOT receive high confidence.

OUTPUT REQUIREMENT

Return ONLY valid JSON. Never return Markdown. Never return explanations. Never return commentary before or after the JSON.

Use this structure:

{"document_type":{"value":null,"confidence":"unreadable"},"surname":{"value":null,"confidence":"unreadable"},"given_names":{"value":null,"confidence":"unreadable"},"date_of_birth":{"value":null,"confidence":"unreadable"},"place_of_birth":{"value":null,"confidence":"unreadable"},"nationality":{"value":null,"confidence":"unreadable"},"sex":{"value":null,"confidence":"unreadable"},"document_number":{"value":null,"confidence":"unreadable"},"issue_date":{"value":null,"confidence":"unreadable"},"expiry_date":{"value":null,"confidence":"unreadable"},"issuing_authority":{"value":null,"confidence":"unreadable"},"mrz":{"value":null,"confidence":"unreadable"},"image_quality":{"overall":"unreadable","reason":null}}

Each field object may additionally contain "uncertain_positions": a list of 1-based character indices that could not be reliably distinguished.

MOST IMPORTANT PRINCIPLE

When forced to choose between (A) an incomplete/uncertain result and (B) a plausible but unsupported result, ALWAYS choose A. False information is worse than missing information. The correct behavior for an unreadable document is null, not a guess. Behave like a forensic transcription system, not like a conversational assistant."""


USER_PROMPT = """Extract the identity information from this document according to the OCR rules in your system instructions.

Read the image directly.
Do not infer or reconstruct anything that is not clearly visible.

For every requested field:
- Locate the corresponding field in the document.
- Read the characters directly from the pixels.
- Verify every character visually.
- If one or more characters cannot be reliably distinguished, do not guess them.
- If the field cannot be reliably read, return null.
- Assign confidence based strictly on visual evidence.

Pay particular attention to: document number, dates, names, visually similar letters and digits, MRZ characters, and characters damaged by blur, compression, low contrast, or scanning artifacts.

Return ONLY the JSON object defined in the system instructions."""

print(f"system prompt: {len(SYSTEM_PROMPT)} chars")

In [ ]:
FIELD_KEYS = ["document_type", "surname", "given_names", "date_of_birth", "place_of_birth",
              "nationality", "sex", "document_number", "issue_date", "expiry_date",
              "issuing_authority", "mrz"]
CONFIDENCES = ["high", "medium", "low", "unreadable"]
CONF_RANK = {"unreadable": 0, "low": 1, "medium": 2, "high": 3}
CORE_FIELDS = ["surname", "given_names", "date_of_birth", "document_number"]

_FIELD_SCHEMA = {
    "type": "object",
    "properties": {
        "value": {"type": ["string", "null"]},
        "confidence": {"type": "string", "enum": CONFIDENCES},
        "uncertain_positions": {"type": "array", "items": {"type": "integer"}},
    },
    "required": ["value", "confidence"],
    "additionalProperties": False,
}

JSON_SCHEMA = {
    "type": "object",
    "properties": {
        **{k: _FIELD_SCHEMA for k in FIELD_KEYS},
        "image_quality": {
            "type": "object",
            "properties": {
                "overall": {"type": "string", "enum": CONFIDENCES},
                "reason": {"type": ["string", "null"]},
            },
            "required": ["overall", "reason"],
            "additionalProperties": False,
        },
    },
    "required": FIELD_KEYS + ["image_quality"],
    "additionalProperties": False,
}


def empty_result(reason: Optional[str] = None) -> Dict[str, Any]:
    """All-null skeleton — the mandated output for anything not readable."""
    r = {k: {"value": None, "confidence": "unreadable"} for k in FIELD_KEYS}
    r["image_quality"] = {"overall": "unreadable", "reason": reason}
    return r


print("schema ready:", len(FIELD_KEYS), "fields")

## 8 · Model backend

`vllm` is the default (continuous batching + prefix caching + grammar-constrained JSON).
A `transformers` backend is provided for workspaces where vLLM cannot be installed.
The first cell also verifies that the checkpoint really is a **vision-language** model —
a text-only checkpoint cannot read a scan and must not be used here.

In [ ]:
def inspect_checkpoint(model_path: str) -> Dict[str, Any]:
    cfg_file = Path(model_path) / "config.json"
    if not cfg_file.exists():
        raise FileNotFoundError(f"config.json not found under {model_path}")
    conf = json.loads(cfg_file.read_text())
    archs = conf.get("architectures", [])
    is_vl = any(("VL" in a) or ("Vision" in a) or ("ImageText" in a) for a in archs)
    info = {"architectures": archs, "model_type": conf.get("model_type"),
            "is_vision_language": is_vl,
            "torch_dtype": conf.get("torch_dtype"),
            "max_position_embeddings": conf.get("max_position_embeddings")}
    print(json.dumps(info, indent=2))
    if not is_vl:
        log.error("This checkpoint does not expose a vision tower. Point CFG.model_path "
                  "at a Qwen*-VL checkpoint — a text-only model cannot read scans.")
    return info


CKPT_INFO = inspect_checkpoint(CFG.model_path)

In [ ]:
class VLLMBackend:
    """Offline vLLM engine: one load, full-corpus batching, schema-constrained JSON."""

    def __init__(self, cfg: Config):
        from vllm import LLM
        from transformers import AutoProcessor

        self.cfg = cfg
        self.processor = AutoProcessor.from_pretrained(
            cfg.model_path, trust_remote_code=True,
            min_pixels=cfg.min_pixels, max_pixels=cfg.max_pixels)

        t0 = time.time()
        self.llm = LLM(
            model=cfg.model_path,
            tokenizer=cfg.model_path,
            trust_remote_code=True,
            dtype=cfg.dtype,
            tensor_parallel_size=cfg.tensor_parallel_size,
            gpu_memory_utilization=cfg.gpu_memory_utilization,
            max_model_len=cfg.max_model_len,
            limit_mm_per_prompt={"image": 1},
            mm_processor_kwargs={"min_pixels": cfg.min_pixels,
                                 "max_pixels": cfg.max_pixels},
            enable_prefix_caching=cfg.enable_prefix_caching,
            disable_log_stats=True,
            seed=0,
        )
        log.info("vLLM engine ready in %.1fs", time.time() - t0)
        self.sampling = self._sampling_params()

    def _sampling_params(self):
        from vllm import SamplingParams
        kw = dict(temperature=0.0, top_p=1.0, max_tokens=self.cfg.max_new_tokens,
                  repetition_penalty=1.0, seed=0)
        if not self.cfg.guided_json:
            return SamplingParams(**kw)
        try:                                              # vLLM >= 0.6.3
            from vllm.sampling_params import GuidedDecodingParams
            return SamplingParams(
                guided_decoding=GuidedDecodingParams(json=JSON_SCHEMA), **kw)
        except Exception as e:
            log.warning("guided decoding unavailable (%s) → free JSON + strict parser", e)
            return SamplingParams(**kw)

    def _prompt(self) -> str:
        msgs = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": [{"type": "image"},
                                             {"type": "text", "text": USER_PROMPT}]}]
        return self.processor.apply_chat_template(msgs, tokenize=False,
                                                  add_generation_prompt=True)

    def generate(self, images: List[Image.Image]) -> List[str]:
        if not images:
            return []
        p = self._prompt()
        reqs = [{"prompt": p, "multi_modal_data": {"image": im}} for im in images]
        outs = self.llm.generate(reqs, sampling_params=self.sampling)
        return [o.outputs[0].text for o in outs]


class TransformersBackend:
    """Fallback backend. Correct but markedly slower — no continuous batching."""

    def __init__(self, cfg: Config, micro_batch: int = 4):
        import torch
        from transformers import AutoProcessor
        try:
            from transformers import AutoModelForImageTextToText as _AM
        except ImportError:
            from transformers import AutoModelForVision2Seq as _AM

        self.cfg, self.micro_batch, self.torch = cfg, micro_batch, torch
        self.processor = AutoProcessor.from_pretrained(
            cfg.model_path, trust_remote_code=True,
            min_pixels=cfg.min_pixels, max_pixels=cfg.max_pixels)
        self.processor.tokenizer.padding_side = "left"
        self.model = _AM.from_pretrained(
            cfg.model_path, torch_dtype=torch.bfloat16, device_map="auto",
            trust_remote_code=True, attn_implementation="sdpa").eval()

    def generate(self, images: List[Image.Image]) -> List[str]:
        outs = []
        for i in range(0, len(images), self.micro_batch):
            chunk = images[i:i + self.micro_batch]
            msgs = [[{"role": "system", "content": SYSTEM_PROMPT},
                     {"role": "user", "content": [{"type": "image"},
                                                  {"type": "text", "text": USER_PROMPT}]}]
                    for _ in chunk]
            texts = [self.processor.apply_chat_template(m, tokenize=False,
                                                        add_generation_prompt=True)
                     for m in msgs]
            inputs = self.processor(text=texts, images=chunk, return_tensors="pt",
                                    padding=True).to(self.model.device)
            with self.torch.inference_mode():
                gen = self.model.generate(**inputs, do_sample=False,
                                          max_new_tokens=self.cfg.max_new_tokens)
            trimmed = [g[len(inp):] for inp, g in zip(inputs.input_ids, gen)]
            outs += self.processor.batch_decode(trimmed, skip_special_tokens=True)
        return outs


def load_backend(cfg: Config):
    if cfg.backend == "vllm":
        try:
            return VLLMBackend(cfg)
        except Exception as e:
            log.warning("vLLM unavailable (%s) → transformers backend", e)
    return TransformersBackend(cfg)


BACKEND = load_backend(CFG)

## 9 · Strict parsing, page merge, MRZ validation

* **Parsing** — anything that is not schema-valid JSON becomes the all-`null` skeleton.
  A malformed answer is never salvaged with regex guesses.
* **Merging** — per field, the highest *visual* confidence wins. Two different values at
  the same confidence level ⇒ `null` + conflict flag (principle: missing beats false).
* **MRZ** — ICAO check digits are computed for **reporting only**. A failed checksum is
  recorded in the sidecar; the transcribed characters are never altered.

In [ ]:
_JSON_RE = re.compile(r"\{.*\}", re.S)


def parse_model_json(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None
    t = text.strip()
    t = re.sub(r"^```(?:json)?|```$", "", t, flags=re.M).strip()
    m = _JSON_RE.search(t)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None


def coerce_result(obj: Any, reason_if_bad: str = "invalid model output") -> Dict[str, Any]:
    """Force any model answer into the exact contract. Unknown keys are dropped."""
    if not isinstance(obj, dict):
        return empty_result(reason_if_bad)
    out = {}
    for k in FIELD_KEYS:
        f = obj.get(k)
        if not isinstance(f, dict):
            out[k] = {"value": None, "confidence": "unreadable"}
            continue
        v = f.get("value")
        if isinstance(v, (int, float)):
            v = str(v)
        if not isinstance(v, str) or not v.strip():
            v = None
        c = f.get("confidence")
        c = c if c in CONFIDENCES else "unreadable"
        if v is None:
            c = "unreadable"                      # null can never carry confidence
        entry = {"value": v, "confidence": c}
        up = f.get("uncertain_positions")
        if isinstance(up, list) and up:
            entry["uncertain_positions"] = [int(i) for i in up
                                            if isinstance(i, (int, float))]
        out[k] = entry
    iq = obj.get("image_quality") if isinstance(obj.get("image_quality"), dict) else {}
    overall = iq.get("overall")
    out["image_quality"] = {
        "overall": overall if overall in CONFIDENCES else "unreadable",
        "reason": iq.get("reason") if isinstance(iq.get("reason"), str) else None,
    }
    return out

In [ ]:
_MRZ_W = [7, 3, 1]


def _mrz_val(ch: str) -> Optional[int]:
    if ch == "<":
        return 0
    if ch.isdigit():
        return int(ch)
    if "A" <= ch <= "Z":
        return ord(ch) - 55
    return None


def mrz_check_digit(s: str) -> Optional[int]:
    tot = 0
    for i, ch in enumerate(s):
        v = _mrz_val(ch)
        if v is None:
            return None
        tot += v * _MRZ_W[i % 3]
    return tot % 10


def validate_mrz(mrz: Optional[str]) -> Dict[str, Any]:
    """Report-only validation. NEVER used to modify or reconstruct characters."""
    if not mrz:
        return {"checked": False}
    lines = [l.strip().replace(" ", "") for l in str(mrz).splitlines() if l.strip()]
    res = {"checked": True, "n_lines": len(lines),
           "line_lengths": [len(l) for l in lines], "checks": {}}
    try:
        if len(lines) == 2 and len(lines[1]) >= 43:            # TD3 passport
            l2 = lines[1]
            res["checks"]["document_number"] = mrz_check_digit(l2[0:9]) == int(l2[9]) \
                if l2[9].isdigit() else None
            res["checks"]["date_of_birth"] = mrz_check_digit(l2[13:19]) == int(l2[19]) \
                if l2[19].isdigit() else None
            res["checks"]["expiry_date"] = mrz_check_digit(l2[21:27]) == int(l2[27]) \
                if l2[27].isdigit() else None
        elif len(lines) == 3:                                   # TD1 ID card
            l1, l2 = lines[0], lines[1]
            if len(l1) >= 15 and l1[14].isdigit():
                res["checks"]["document_number"] = mrz_check_digit(l1[5:14]) == int(l1[14])
            if len(l2) >= 15:
                if l2[6].isdigit():
                    res["checks"]["date_of_birth"] = mrz_check_digit(l2[0:6]) == int(l2[6])
                if l2[14].isdigit():
                    res["checks"]["expiry_date"] = mrz_check_digit(l2[8:14]) == int(l2[14])
    except Exception as e:
        res["error"] = str(e)
    vals = [v for v in res["checks"].values() if v is not None]
    res["all_valid"] = bool(vals) and all(vals)
    res["note"] = "validation is informational only; characters are never repaired"
    return res

In [ ]:
def merge_pages(page_results: List[Dict[str, Any]]) -> Tuple[Dict[str, Any], List[Dict]]:
    """Field-wise merge across pages, driven strictly by reported visual confidence."""
    if not page_results:
        return empty_result("no candidate page"), []

    merged, conflicts = {}, []
    for k in FIELD_KEYS:
        best, best_rank, seen = None, -1, []
        for idx, pr in enumerate(page_results):
            f = pr.get(k, {})
            v, c = f.get("value"), f.get("confidence", "unreadable")
            if v is None:
                continue
            seen.append({"page_index": idx, "value": v, "confidence": c})
            r = CONF_RANK.get(c, 0)
            if r > best_rank:
                best, best_rank = dict(f), r
        if best is None:
            merged[k] = {"value": None, "confidence": "unreadable"}
            continue
        rivals = {s["value"] for s in seen
                  if CONF_RANK.get(s["confidence"], 0) == best_rank}
        if len(rivals) > 1:
            # equal visual evidence, different readings → we cannot know. Return null.
            conflicts.append({"field": k, "candidates": seen,
                              "resolution": "null (equal-confidence disagreement)"})
            merged[k] = {"value": None, "confidence": "unreadable"}
        else:
            merged[k] = best

    worst = min((CONF_RANK.get(p.get("image_quality", {}).get("overall", "unreadable"), 0)
                 for p in page_results), default=0)
    best_q = max((CONF_RANK.get(p.get("image_quality", {}).get("overall", "unreadable"), 0)
                  for p in page_results), default=0)
    reasons = [p.get("image_quality", {}).get("reason") for p in page_results
               if p.get("image_quality", {}).get("reason")]
    merged["image_quality"] = {
        "overall": {v: k for k, v in CONF_RANK.items()}[best_q],
        "reason": "; ".join(dict.fromkeys(reasons))[:500] or None,
    }
    _ = worst
    return merged, conflicts


def needs_retry(merged: Dict[str, Any]) -> bool:
    """Wave-2 trigger: any core field still missing or only weakly supported."""
    for k in CORE_FIELDS:
        f = merged.get(k, {})
        if f.get("value") is None or CONF_RANK.get(f.get("confidence"), 0) <= 1:
            return True
    return False


print("merge/validation ready")

## 10 · Extraction run (wave 1 → wave 2)

In [ ]:
def build_wave(prepared: Dict[str, Dict[str, Any]], customers: List[str], cfg: Config,
               k: int, rotate180: bool = False, max_pixels: Optional[int] = None
               ) -> Tuple[List[Image.Image], List[Dict[str, Any]]]:
    """Pre-process the top-k pages of each customer and return a flat GPU batch."""
    images, index = [], []
    budget = max_pixels or cfg.max_pixels
    for cust in customers:
        prep = prepared.get(cust, {})
        for rank, page in enumerate(prep.get("pages", [])[:k]):
            try:
                pil, meta = preprocess_page(page["page_path"], cfg)
                if rotate180:
                    pil = pil.rotate(180, expand=True)
                    meta["rotation_applied"] = (meta.get("rotation_applied", 0) + 180) % 360
                pil = fit_pixel_budget(pil, budget)
                meta.update({"customer_id": cust, "rank": rank,
                             "triage_score": page.get("score"),
                             "sent_size": pil.size})
                images.append(pil)
                index.append(meta)
            except Exception as e:
                log.warning("preprocess failed %s: %s", page["page_path"], e)
    return images, index


def run_wave(backend, images, index) -> Dict[str, List[Dict[str, Any]]]:
    if not images:
        return {}
    t0 = time.time()
    raw = backend.generate(images)                     # one batched call
    dt = time.time() - t0
    log.info("wave: %d pages in %.1fs (%.2fs/page)", len(images), dt, dt / len(images))

    by_customer = defaultdict(list)
    for meta, txt in zip(index, raw):
        parsed = parse_model_json(txt)
        result = coerce_result(parsed, "model returned non-JSON or invalid JSON")
        result["_meta"] = {**meta, "raw_len": len(txt or ""),
                           "parse_ok": parsed is not None}
        by_customer[meta["customer_id"]].append(result)
    return dict(by_customer)


def run_extraction(cfg: Config, prepared: Dict[str, Dict[str, Any]], backend):
    stats = {"t_start": time.time()}
    customers = sorted(prepared)

    # ── wave 1: top-k triaged pages, standard resolution ─────────────────────
    imgs, idx = build_wave(prepared, customers, cfg, cfg.top_k_pages)
    stats["wave1_pages"] = len(imgs)
    w1 = run_wave(backend, imgs, idx)

    per_customer = {c: list(w1.get(c, [])) for c in customers}
    merged = {c: merge_pages([{k: v for k, v in r.items() if k != "_meta"}
                              for r in per_customer[c]])[0] for c in customers}

    # ── wave 2: only the customers still incomplete ──────────────────────────
    stats["wave2_pages"] = 0
    if cfg.enable_wave2_retry:
        retry = [c for c in customers if needs_retry(merged[c])]
        stats["wave2_customers"] = len(retry)
        if retry:
            log.info("wave 2: %d customers incomplete → 180° + high-res retry", len(retry))
            # (a) remaining pages at higher resolution
            imgs_a, idx_a = build_wave(prepared, retry, cfg,
                                       k=cfg.max_pages_per_pdf,
                                       max_pixels=cfg.max_pixels_retry)
            # keep only pages not already processed in wave 1
            done = {(m["customer_id"], m["page_path"]) for m in idx}
            keep = [(im, m) for im, m in zip(imgs_a, idx_a)
                    if (m["customer_id"], m["page_path"]) not in done]
            # (b) 180°-flipped top page (covers upside-down scans when OSD is absent)
            imgs_b, idx_b = build_wave(prepared, retry, cfg, k=1, rotate180=True,
                                       max_pixels=cfg.max_pixels_retry)
            keep += list(zip(imgs_b, idx_b))
            if keep:
                imgs2 = [x[0] for x in keep]
                idx2 = [x[1] for x in keep]
                stats["wave2_pages"] = len(imgs2)
                w2 = run_wave(backend, imgs2, idx2)
                for c, res in w2.items():
                    per_customer[c].extend(res)

    # ── final merge ──────────────────────────────────────────────────────────
    final, conflicts_all = {}, {}
    for c in customers:
        clean = [{k: v for k, v in r.items() if k != "_meta"} for r in per_customer[c]]
        m, conf = merge_pages(clean)
        final[c], conflicts_all[c] = m, conf

    stats["t_total"] = round(time.time() - stats.pop("t_start"), 2)
    stats["customers"] = len(customers)
    return final, per_customer, conflicts_all, stats


FINAL, PER_PAGE, CONFLICTS, STATS = run_extraction(CFG, PREPARED, BACKEND)
print(json.dumps(STATS, indent=2))

## 11 · Persist results

* `extractions/<customer>/identity_extraction.json` — **exactly** the contract structure,
  nothing added.
* `_debug/<customer>.json` — page-level answers, pre-processing metadata, MRZ validation,
  conflicts. Engineering metadata is kept strictly out of the deliverable JSON.

In [ ]:
def persist(cfg: Config, final, per_page, conflicts, stats) -> pd.DataFrame:
    rows = []
    for cust, res in final.items():
        d = cfg.out_dir / cust
        d.mkdir(parents=True, exist_ok=True)
        (d / "identity_extraction.json").write_text(
            json.dumps(res, indent=2, ensure_ascii=False), encoding="utf-8")

        (cfg.debug_dir / f"{cust}.json").write_text(json.dumps({
            "customer_id": cust,
            "source_pdf": TARGETS.get(cust),
            "triage": {k: v for k, v in PREPARED.get(cust, {}).items() if k != "pages"},
            "pages_considered": [p.get("page_path") for p in PREPARED.get(cust, {}).get("pages", [])],
            "page_results": per_page.get(cust, []),
            "conflicts": conflicts.get(cust, []),
            "mrz_validation": validate_mrz(res["mrz"]["value"]),
        }, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

        row = {"customer_id": cust}
        for k in FIELD_KEYS:
            row[k] = res[k]["value"]
            row[f"{k}__confidence"] = res[k]["confidence"]
        row["image_quality"] = res["image_quality"]["overall"]
        row["image_quality_reason"] = res["image_quality"]["reason"]
        row["n_pages_seen"] = len(per_page.get(cust, []))
        row["n_conflicts"] = len(conflicts.get(cust, []))
        row["needs_human_review"] = needs_retry(res) or bool(conflicts.get(cust))
        rows.append(row)

    df = pd.DataFrame(rows).sort_values("customer_id").reset_index(drop=True)
    df.to_csv(cfg.reports_dir / "identity_extractions.csv", index=False)
    with open(cfg.reports_dir / "identity_extractions.jsonl", "w", encoding="utf-8") as f:
        for cust, res in sorted(final.items()):
            f.write(json.dumps({"customer_id": cust, **res}, ensure_ascii=False) + "\n")
    (cfg.reports_dir / "run_report.json").write_text(json.dumps({
        "stats": stats,
        "model_path": cfg.model_path,
        "checkpoint": CKPT_INFO,
        "config": {k: str(v) for k, v in vars(cfg).items()},
        "customers_total": len(INVENTORY),
        "customers_target": len(TARGETS),
        "seconds_per_customer": round(stats["t_total"] / max(1, len(final)), 2),
    }, indent=2, ensure_ascii=False), encoding="utf-8")
    return df


RESULTS = persist(CFG, FINAL, PER_PAGE, CONFLICTS, STATS)
print(f"written to {CFG.work_dir}")
RESULTS.head(10)

## 12 · Quality dashboard

Field fill-rate and confidence distribution. `needs_human_review` is the queue an operator
should actually work: everything the pipeline refused to guess lands there, by design.

In [ ]:
conf_cols = [f"{k}__confidence" for k in FIELD_KEYS]
summary = pd.DataFrame({
    "filled": [RESULTS[k].notna().sum() for k in FIELD_KEYS],
    "high": [(RESULTS[f"{k}__confidence"] == "high").sum() for k in FIELD_KEYS],
    "medium": [(RESULTS[f"{k}__confidence"] == "medium").sum() for k in FIELD_KEYS],
    "low": [(RESULTS[f"{k}__confidence"] == "low").sum() for k in FIELD_KEYS],
    "unreadable": [(RESULTS[f"{k}__confidence"] == "unreadable").sum() for k in FIELD_KEYS],
}, index=FIELD_KEYS)
summary["fill_rate"] = (summary["filled"] / max(1, len(RESULTS))).round(3)

print(f"target customers processed : {len(RESULTS)}")
print(f"needs human review         : {int(RESULTS.needs_human_review.sum())} "
      f"({RESULTS.needs_human_review.mean():.1%})")
print(f"total inference wall time  : {STATS['t_total']}s "
      f"({STATS['wave1_pages'] + STATS['wave2_pages']} pages)")
summary

In [ ]:
# ── Single-document debug helper ─────────────────────────────────────────────
def debug_customer(customer_id: str, show_pages: int = 1):
    print(json.dumps(FINAL[customer_id], indent=2, ensure_ascii=False))
    print("\nMRZ validation:",
          json.dumps(validate_mrz(FINAL[customer_id]["mrz"]["value"]), indent=2))
    if CONFLICTS.get(customer_id):
        print("\nConflicts:", json.dumps(CONFLICTS[customer_id], indent=2, ensure_ascii=False))
    for p in PREPARED.get(customer_id, {}).get("pages", [])[:show_pages]:
        pil, meta = preprocess_page(p["page_path"], CFG)
        print("\npre-processing:", json.dumps(meta, default=str))
        display(fit_pixel_budget(pil, 900 * 900))


# debug_customer(list(FINAL)[0])
print("run  debug_customer('<customer_id>')  to inspect one folder")

## Operating notes

**Tuning latency.** Measure first, then move one knob:
`max_pixels` (vision tokens scale linearly with it) → `top_k_pages` → `max_new_tokens`.
Disabling `use_tesseract_triage` removes ~150 ms of CPU per page but sends more pages to
the GPU; it is a net loss on documents with more than ~3 pages.

**If the checkpoint is not a VL model.** `inspect_checkpoint` will say so. A text-only
Qwen cannot read a scan; no prompt engineering compensates for a missing vision tower.
Repoint `CFG.model_path` at the VL checkpoint in the model hub.

**Arabic / mixed-script documents.** No script conversion is performed anywhere in the
pipeline — Arabic stays Arabic, Latin stays Latin, and a name present in both scripts is
transcribed as it appears. Nothing is transliterated.

**What this pipeline deliberately does not do.** It does not repair MRZ checksums, does
not normalise date formats, does not pattern-match document numbers into an expected
format, and does not fill a field because the document type implies it should exist. Every
one of those would convert a legible gap into an invisible error.

**Review queue.** `needs_human_review = True` is the intended, healthy output for a
degraded scan. Track its rate over time: a sudden drop usually means scanner settings
changed; a sudden rise usually means a batch of bad scans, not a model regression.